# GRPO Reinforcement Learning - Chatbot Tim Legal (PGABL) [Advanced / Opsional]
**Nama:** Stanley Nathanael Wijaya

Notebook ini melanjutkan model hasil `Fine-tuning_submission_PGABL_Stanley-Nathanael-Wijaya.ipynb`
dengan **GRPO (Group Relative Policy Optimization)** memakai `GRPOTrainer` dari TRL + Unsloth, agar
model belajar menampilkan proses berpikir (`<think>...</think>`) sebelum menjawab.

**PENTING:** GRPO membangkitkan banyak *completion* per prompt (`num_generations`) sehingga jauh
lebih berat dari SFT biasa. Jalankan di Colab/Kaggle dengan GPU >= T4 16GB. Parameter
`num_generations` dan `max_completion_length` sengaja dibuat kecil untuk memitigasi OOM.


## 1. Instalasi & Load Model Hasil Fine-tuning

In [ ]:
%%capture
import importlib
if importlib.util.find_spec("unsloth") is None:
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-deps trl peft accelerate bitsandbytes xformers
!pip install -q datasets wandb rouge-score

In [ ]:
import os
import re
from getpass import getpass

import torch
from huggingface_hub import login

SEED = 3407
torch.manual_seed(SEED)

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("Masukkan Hugging Face Write Token: ")
login(token=HF_TOKEN)

HF_USERNAME = os.environ.get("HF_USERNAME") or input("Masukkan username Hugging Face kamu: ")
FT_REPO_ID = os.environ.get("FT_REPO_ID") or f"{HF_USERNAME}/qwen2.5-1.5b-legal-chatbot-id"
GRPO_REPO_ID = f"{HF_USERNAME}/qwen2.5-1.5b-legal-chatbot-id-grpo"
print("Memuat kembali model instruct hasil fine-tuning dari:", FT_REPO_ID)

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=FT_REPO_ID,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

## 2. Dataset & System Prompt

Menggunakan dataset yang sama (`Ichsan2895/alpaca-gpt4-indonesian`) sesuai ketentuan submission
(dataset untuk fine-tuning **dan** GRPO wajib dataset yang sama). Prompt sistem diarahkan agar model
selalu berpikir di dalam tag `<think>...</think>` sebelum menjawab.


In [ ]:
from datasets import load_dataset

GRPO_SYSTEM_PROMPT = (
    "Kamu adalah asisten AI internal Tim Legal perusahaan. Sebelum menjawab, tuliskan proses "
    "berpikirmu di dalam tag <think>...</think>, lalu berikan jawaban akhir dalam Bahasa Indonesia "
    "berdasarkan konteks/pertanyaan yang diberikan."
)

raw_dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")


def normalize_to_alpaca(example):
    return {"instruction": example["input"], "input": "", "output": example["output"]}


if "instruction" not in raw_dataset.column_names:
    raw_dataset = raw_dataset.map(normalize_to_alpaca, remove_columns=raw_dataset.column_names)

# Subset agar eksperimen GRPO tidak berjalan terlalu lama
grpo_dataset = raw_dataset.shuffle(seed=SEED).select(range(2000))


def to_grpo_prompt(example):
    prompt = [
        {"role": "system", "content": GRPO_SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
    ]
    return {
        "prompt": tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True),
        "ground_truth": example["output"],
    }


grpo_dataset = grpo_dataset.map(to_grpo_prompt)
print(grpo_dataset[0]["prompt"])

## 3. Reward Functions

Empat reward function sesuai ketentuan kriteria Advanced.


In [ ]:
THINK_RE = re.compile(r"<think>(.*?)</think>", re.DOTALL)


def _think_matches(text):
    return THINK_RE.findall(text)


def format_reward_func(completions, **kwargs):
    """Reward shaping bertahap (maks +1.0), penalti -0.5 jika tag <think>/</think> duplikat."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        score = 0.0
        opens = text.count("<think>")
        closes = text.count("</think>")

        if opens >= 1:
            score += 0.2
        if closes >= 1:
            score += 0.3

        matches = _think_matches(text)
        starts_with_think = text.strip().startswith("<think>")
        well_closed = len(matches) == 1 and "</think>" in text
        followed_by_answer = bool(re.search(r"</think>\s*\S+", text, re.DOTALL))
        if starts_with_think and well_closed and followed_by_answer:
            score = 1.0

        if opens > 1 or closes > 1:
            score -= 0.5

        rewards.append(score)
    return rewards


def reasoning_length_reward(completions, **kwargs):
    """Poin proporsional berdasarkan panjang isi <think>...</think>."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        matches = _think_matches(text)
        if not matches:
            # toleran terhadap think yang terpotong batas token (tag <think> ada, </think> belum muncul)
            if "<think>" in text:
                content = text.split("<think>", 1)[1].strip()
            else:
                rewards.append(0.0)
                continue
        else:
            content = matches[0].strip()

        if not content:
            rewards.append(0.0)
        elif len(content) < 50:
            rewards.append(0.2)
        elif len(content) < 200:
            rewards.append(0.5)
        else:
            rewards.append(1.0)
    return rewards


def _final_answer(text):
    return THINK_RE.sub("", text).strip()


def correctness_reward(completions, ground_truth, **kwargs):
    """+1.0 jika jawaban akhir mengandung/mirip ground truth (ROUGE-L)."""
    from rouge_score import rouge_scorer

    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    rewards = []
    for completion, truth in zip(completions, ground_truth):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        answer = _final_answer(text)
        if not answer:
            rewards.append(0.0)
            continue
        if truth.strip() and truth.strip() in answer:
            rewards.append(1.0)
            continue
        score = scorer.score(truth, answer)["rougeL"].fmeasure
        rewards.append(1.0 if score >= 0.5 else score)
    return rewards


ENGLISH_HINT_RE = re.compile(r"\b(the|is|are|and|of|to|this|that|with)\b", re.IGNORECASE)


def language_reward_func(completions, **kwargs):
    """-0.5 jika terdeteksi Bahasa Inggris, +1.0 jika murni Bahasa Indonesia."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        answer = _final_answer(text)
        english_hits = len(ENGLISH_HINT_RE.findall(answer))
        rewards.append(-0.5 if english_hits >= 3 else 1.0)
    return rewards

## 4. GRPOTrainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="outputs/grpo",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,          # kecil supaya menghindari OOM di GPU terbatas
    max_completion_length=256,  # dibatasi untuk mengontrol VRAM
    max_prompt_length=512,
    max_steps=200,
    logging_steps=5,
    save_steps=100,
    seed=SEED,
    report_to="none",
)

grpo_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,
        reasoning_length_reward,
        correctness_reward,
        language_reward_func,
    ],
    args=grpo_config,
    train_dataset=grpo_dataset,
)
grpo_trainer.train()

## 5. Push Model GRPO ke Hugging Face Hub

In [ ]:
model.push_to_hub_merged(
    GRPO_REPO_ID,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN,
)
print("Model GRPO berhasil di-push ke:", f"https://huggingface.co/{GRPO_REPO_ID}")

with open("link_huggingface.txt", "a") as f:
    f.write(f"GRPO model: https://huggingface.co/{GRPO_REPO_ID}\n")

## 6. Test Case Wajib

Prompt: *"Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang
lembur?"* — output harus menampilkan proses reasoning di dalam tag `<think>`.


In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = [
    {"role": "system", "content": GRPO_SYSTEM_PROMPT},
    {"role": "user", "content": (
        "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. "
        "Apakah saya berhak dapat uang lembur?"
    )},
]
inputs = tokenizer.apply_chat_template(
    test_prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

outputs = model.generate(input_ids=inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
result = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(result)
assert "<think>" in result, "Model belum menampilkan tag <think> - pertimbangkan menambah steps GRPO."